# MedResearch GPT — training on a free GPU

Works on **Google Colab** (free T4) or **Kaggle Notebooks** (free T4 x2 / P100).

| | Colab free | Kaggle |
|---|---|---|
| GPU | 1x T4 16GB | 1x P100 or 2x T4 |
| Budget | no published quota, throttled after heavy use | **30 GPU-hours/week**, stated up front |
| Max session | ~12h, idle-disconnects aggressively | 9h, survives browser close |
| Persistence | must mount Drive | `/kaggle/working` is saved as output |

**Kaggle is the better default here** — the quota is explicit and a closed laptop
does not kill the run. Colab is easier only if you already keep this project in Drive.

> **Enable the GPU first.** Colab: Runtime -> Change runtime type -> T4 GPU.
> Kaggle: Settings -> Accelerator -> GPU. Cell 1 fails loudly if you forget,
> because a CPU run looks identical apart from being ~40x slower.

In [ ]:
# 1. Confirm there is actually a GPU attached.
import torch, subprocess
assert torch.cuda.is_available(), (
    "No GPU. Colab: Runtime > Change runtime type > T4 GPU. "
    "Kaggle: Settings > Accelerator > GPU. Do not train without this.")
print(torch.cuda.get_device_name(0))
print(f"bf16 supported: {torch.cuda.is_bf16_supported()}  (False on T4/P100 -> fp16 + GradScaler)")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

In [ ]:
# 2. Get the code and the corpus.
#
# --depth 1 matters: the full history is ~210MB because old model checkpoints
# and mlruns/ artifacts were committed. A shallow clone pulls only the current
# tree, which still includes medical_text.txt (58MB) -- the thing you need.
!git clone --depth 1 https://github.com/rajagowthamr/MedResearch.git
%cd MedResearch

# torch is preinstalled on both platforms; only mlflow is missing.
!pip install -q mlflow

import os
print("corpus:", os.path.getsize("medical_text.txt") / 1e6, "MB")
print("checkpoints:", os.listdir("checkpoints"))

## 3. Persistence — do this BEFORE training, not after

The notebook's disk is deleted when the session ends, and a free session can end
without warning. A 6-hour run that saved only to the local disk is a 6-hour run
you lose. Pick the cell for your platform.

In [ ]:
# 3a. COLAB — mount Drive and symlink checkpoints/ into it, so every
#     save_checkpoint() call writes straight to Drive as training proceeds.
#     A disconnect then costs you at most one eval_interval of progress.
from google.colab import drive
drive.mount("/content/drive")

import os, shutil
SAVE = "/content/drive/MyDrive/MedResearch"
os.makedirs(f"{SAVE}/checkpoints", exist_ok=True)
if os.path.isdir("checkpoints") and not os.path.islink("checkpoints"):
    for f in os.listdir("checkpoints"):            # seed Drive with the v2/v3 weights
        dst = f"{SAVE}/checkpoints/{f}"
        if not os.path.exists(dst):
            shutil.copy(f"checkpoints/{f}", dst)
    shutil.rmtree("checkpoints")
    os.symlink(f"{SAVE}/checkpoints", "checkpoints")
print("checkpoints ->", os.path.realpath("checkpoints"))

In [ ]:
# 3b. KAGGLE — /kaggle/working persists as the notebook's output (20GB), so a
#     symlink there survives the session. Download it from the Output tab, or
#     with the Kaggle API, once the run finishes.
import os, shutil
SAVE = "/kaggle/working"
os.makedirs(f"{SAVE}/checkpoints", exist_ok=True)
if os.path.isdir("checkpoints") and not os.path.islink("checkpoints"):
    for f in os.listdir("checkpoints"):
        dst = f"{SAVE}/checkpoints/{f}"
        if not os.path.exists(dst):
            shutil.copy(f"checkpoints/{f}", dst)
    shutil.rmtree("checkpoints")
    os.symlink(f"{SAVE}/checkpoints", "checkpoints")
print("checkpoints ->", os.path.realpath("checkpoints"))

## 4. Pick what "train further" means

Two genuinely different options — the second is the one worth the GPU:

**A. Same model, more steps (resume).** Continues `v2-best` from val 1.2296.
Cheap and safe, but v2 finished on a *cosine schedule that had already decayed
to `min_lr`* — it was close to converged for its size, so expect a modest gain,
not a transformation. The architecture is the ceiling here.

**B. Bigger model from scratch.** 4.86M params exists only because an M-series
Mac made anything larger painful. A T4 has 16GB and real tensor cores, so
`n_embd=512, n_layer=8, block_size=512` (~30M params) is a comfortable fit and
is where the actual quality jump lives. This **cannot** resume from v2 — the
tensor shapes differ, and `train.py` will refuse rather than silently corrupt
the embeddings.

Both are driven by env vars, so neither needs a file edit.

In [ ]:
# 4A. RESUME the existing 4.86M model for another 15k steps.
#
# warmup_iters is deliberately non-zero: save_checkpoint() stores weights but
# NOT AdamW's moment estimates, so those restart from zero. Stepping straight
# in at full LR with empty moments kicks a converged model backwards.
!VERSION=v4-resumed \
 RESUME=checkpoints/gpt_medical_v2-best.pt \
 MAX_ITERS=15000 \
 LEARNING_RATE=3e-4 \
 MIN_LR=3e-5 \
 WARMUP_ITERS=300 \
 BATCH_SIZE=64 \
 EVAL_INTERVAL=500 \
 python train.py

In [ ]:
# 4B. BIGGER model, from scratch. ~30M params, 2x the context window.
#
# Sizing notes for a 16GB T4:
#   block_size 512 quadruples attention cost vs 256 -- the expensive knob.
#   batch_size 64 at block 512 is ~33k tokens/step and fits comfortably.
#   If you hit CUDA OOM, halve BATCH_SIZE before touching the architecture.
# Budget ~3-5h on a T4. Start it, check back; the best checkpoint is written to
# Drive/working at every eval, so an early disconnect still leaves you a model.
!VERSION=v5-large \
 MAX_ITERS=20000 \
 BLOCK_SIZE=512 \
 N_EMBD=512 \
 N_HEAD=8 \
 N_LAYER=8 \
 BATCH_SIZE=64 \
 DROPOUT=0.15 \
 LEARNING_RATE=6e-4 \
 MIN_LR=6e-5 \
 WARMUP_ITERS=500 \
 EVAL_INTERVAL=500 \
 python train.py

In [ ]:
# 5. Sanity-check the result here, before carrying it home.
import torch, glob
from model import load_checkpoint

path = sorted(glob.glob("checkpoints/*.pt"))[-1]
model, cfg, stoi, itos, meta = load_checkpoint(path, "cuda")
print(f"{path}: {model.n_params()/1e6:.2f}M params, step {meta.get('iter')}, "
      f"val {meta.get('val_loss'):.4f} (v2-best was 1.2296)")

unk = stoi.get("\ufffd", 0)
prompt = "Type 2 diabetes mellitus is a chronic metabolic disorder characterised by"
idx = torch.tensor([[stoi.get(c, unk) for c in prompt]], device="cuda")
out = prompt + "".join(itos[t.item()] for t in model.stream(idx, 400, 0.8, 40))
print("\n" + out)

In [ ]:
# 6. Copy the MLflow run database out too, so the loss curves come home with
#    the weights and the new run shows up next to your local ones in `mlflow ui`.
#    checkpoints/ is already a symlink into SAVE, so only mlflow.db needs moving.
import shutil, os
if os.path.exists("mlflow.db"):
    shutil.copy("mlflow.db", f"{SAVE}/mlflow-gpu.db")
    print(f"wrote {SAVE}/mlflow-gpu.db ({os.path.getsize('mlflow.db')/1e6:.1f} MB)")
print(os.listdir(f"{SAVE}/checkpoints"))

# On Colab you can also pull a single checkpoint straight down:
#   from google.colab import files; files.download("checkpoints/gpt_medical_v5-large.pt")
# On Kaggle, use the Output tab in the right-hand panel.

## 7. Back on your Mac

```bash
# drop the downloaded .pt into checkpoints/, then:
source venv/bin/activate
streamlit run gpt_app.py      # the new version appears in the sidebar dropdown
```

Use the **Compare versions** tab to prove the GPU run actually beat v2 on top-1
accuracy. Note the caveat already on that tab: perplexity is not comparable
across different vocab sizes, but these runs all share v2's 249-char tokenizer,
so here it is a fair comparison.

The `mlflow-gpu.db` file is a separate tracking store. To browse it:
`mlflow ui --backend-store-uri sqlite:///mlflow-gpu.db`